In [11]:

import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from task1.classifier import MnistClassifier

In [ ]:
import gradio as gr
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance

# 1. Завантажуємо наші моделі
rf_clf = MnistClassifier(algorithm='rf')
rf_clf.load("../weights/rf_model.joblib")

ffnn_clf = MnistClassifier(algorithm='ffnn')
ffnn_clf.load("../weights/ffnn_model.pth")

cnn_clf = MnistClassifier(algorithm='cnn')
cnn_clf.load("../weights/cnn_model.pth")


from pathlib import Path
import time

def predict_digit(image):
    if image is None: 
        return "Draw some digit", None
    
    img = Image.fromarray(image["composite"]).convert('RGBA')
    bg = Image.new('RGBA', img.size, (255, 255, 255))
    combined = Image.alpha_composite(bg, img).convert('L')
    inverted_img = ImageOps.invert(combined)
    processed_img = inverted_img.resize((28, 28), Image.Resampling.LANCZOS)
    processed_img = processed_img.point(lambda p: p * 1.5 if p > 50 else 0)

    debug_dir = Path("debug_inputs")
    debug_dir.mkdir(exist_ok=True)
  
    save_path = debug_dir / "last_input.png"
    processed_img.save(save_path)
    
    img_array = np.array(processed_img).astype('float32') / 255.0
    input_data = img_array[None, ...]

    rf_res = rf_clf.predict(input_data)[0]
    ffnn_res = ffnn_clf.predict(input_data)[0]
    cnn_res = cnn_clf.predict(input_data)[0]

    return f"RF: {rf_res}; FFNN: {ffnn_res}; CNN: {cnn_res}"

interface = gr.Interface(
    fn=predict_digit,
    inputs=gr.Sketchpad(label="Draw a digit here", type="numpy"),
    outputs=gr.Textbox(label="Model Predictions"),
    title="MNIST Digit Classifiers",
    live=True
)

interface.launch()

Weights loaded from ../weights/ffnn_model.pth
Weights loaded from ../weights/cnn_model.pth
* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
